# 12b — View Cutouts for dipole-rich diaObjects

## Purpose

Visualise the triplet of image stamps **(Science / Template / Difference / Science−Template)**
for every diaSource detection of a user-selected `diaObjectId`, in order to demonstrate
that **dipole residuals** in the PSF-subtracted difference image are responsible for
triggering spurious Rubin AP alerts.

The objects are selected from the dipole-rich catalogue produced by `03b_dipoleobjectcorr.ipynb`.

## Layout — 2×3 compact grid per diaSource (presentation-ready)

| Position | Content |
|----------|---------|
| (0,0) | **Info panel** — object metadata, fluxes, dipole flags + geometry, dates |
| (0,1) | **Science** image — shared vmin/vmax with Template |
| (0,2) | **Template** image — shared vmin/vmax with Science |
| (1,0) | **Light curve** in the current band (src+fp, current visit highlighted) |
| (1,1) | **DIA Difference** (downloaded) — symmetric ±vmax, diverging colormap |
| (1,2) | **Sci − Template** (computed) — same ±vmax, diverging colormap |

## Data sources

| Data | Location |
|------|----------|
| Cutout `.npy` arrays + all metadata (incl. dipole flags) | `fullcutouts_{diaObjectId}/manifest.csv` |
| Forced photometry | `fullcutouts_{diaObjectId}/manifest_fp.csv` |
| Dipole-rich object catalogue (stats + geometry) | `data_DIPOLES_03b/topranked_objects_dipoles.csv` |
| Dipole angle stability | `data_DIPOLES_03b/dipole_angle_stability.csv` |

## Key dipole columns in the manifest

| Column | Description |
|--------|-------------|
| `r:isDipole` | Flag: source classified as a dipole |
| `r:isNegative` | Flag: source is the negative lobe of a dipole |
| `r:dipoleFitAttempted` | Flag: dipole fit was attempted |
| `r:dipoleFluxDiff` | Flux difference between dipole lobes (nJy) |
| `r:dipoleFluxDiffErr` | Uncertainty on dipoleFluxDiff (nJy) |
| `r:dipoleMeanFlux` | Mean flux of the two dipole lobes (nJy) |
| `r:dipoleMeanFluxErr` | Uncertainty on dipoleMeanFlux (nJy) |
| `r:dipoleLength` | Separation between dipole lobes (arcsec) |
| `r:dipoleAngle` | Position angle of the dipole axis (deg, N→E) |
| `r:dipoleNdata` | Number of pixels used in the dipole fit |
| `r:dipoleChi2` | Chi² of the dipole fit |

---
- **Author:** Sylvie Dagoret-Campagne — IJCLab / IN2P3 / CNRS — Université Paris-Saclay
- **Creation date:** 2026-05-13
- **Last update:** 2026-05-27 — adapted for dipole-rich objects from 03b; dipole geometry variables
- **Subject:** Fink/LSST DIA — Dipole hypothesis — cutout visualisation


## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize

from astropy.time import Time

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
%matplotlib inline

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# USER PARAMETERS  ← edit here
# ─────────────────────────────────────────────────────────────────────────────

# diaObjectId to inspect — must have a fullcutouts_{id}/ directory
# (produced by notebook 12a).  The dict maps a short index to each id
# for convenience; set DIAOBJECT_IDX to switch objects without hunting
# for the full 18-digit identifier.
# Top-ranked dipole-rich objects from data_DIPOLES_03b/topranked_objects_dipoles.csv:
objsid = {
    # 0: 313888627167330394,   # rank 1  COSMOS  dipole_frac=0.962  n_dipoles=507
    0: 170019717267849383,
    1: 313985344866353157,  # rank 2  COSMOS  dipole_frac=0.978  n_dipoles=493
    2: 313853517840777344,  # rank 3  COSMOS  dipole_frac=0.958  n_dipoles=461
    3: 313972182542712999,  # rank 4  COSMOS  dipole_frac=0.918  n_dipoles=416
    4: 313994141002367046,
}

DIAOBJECT_IDX = 4  # ← change index to switch object
DIAOBJECT_ID = objsid[DIAOBJECT_IDX]

# ── MJD window filter (None = show all downloaded diaSources) ─────────────────
# Useful to restrict the display to a specific observing period.
MJD_MIN = None  # float or None
MJD_MAX = None  # float or None

# Bands to display: None = all available, or e.g. ["r", "i"]
BANDS_FILTER = None

# Max diaSources per band to display (None = unlimited)
MAX_ROWS_PER_BAND = None

# Image stretch for science/template panels: percentile-based z-scale
STRETCH_MODE = "zscale"  # only mode currently supported

# Color maps
CMAP_SCI = "gray"
CMAP_DIFF = "RdBu_r"  # diverging: red = positive, blue = negative

# Save figures?
SAVE_FIGS = True
DIR_FIGS = "figs_DIPOLES_12b"

# ─────────────────────────────────────────────────────────────────────────────
# Derived paths  (do not edit below this line)
# ─────────────────────────────────────────────────────────────────────────────
DIR_CUTOUTS = f"fullcutouts_{DIAOBJECT_ID}"
FILE_MANIFEST = os.path.join(DIR_CUTOUTS, "manifest.csv")
FILE_MANIFEST_FP = os.path.join(DIR_CUTOUTS, "manifest_fp.csv")

# Object-level dipole statistics from notebook 03b
FILE_DIPOLE_STATS = os.path.join("data_DIPOLES_03b", "topranked_objects_dipoles.csv")
FILE_DIPOLE_ANGLES = os.path.join("data_DIPOLES_03b", "dipole_angle_stability.csv")

# Photometric constants
AB_FLUX_ZERO = 3631e9  # nJy → m_AB = −2.5 × log10(f / AB_FLUX_ZERO)
BAND_ORDER = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

os.makedirs(DIR_FIGS, exist_ok=True)

print(f"diaObjectId  : {DIAOBJECT_ID}  (index {DIAOBJECT_IDX})")
print(f"Manifest     : {os.path.abspath(FILE_MANIFEST)}")
print(f"Manifest fp  : {os.path.abspath(FILE_MANIFEST_FP)}")
print(f"Dipole stats : {os.path.abspath(FILE_DIPOLE_STATS)}")
print(f"Figures      : {os.path.abspath(DIR_FIGS)}")
if MJD_MIN is not None or MJD_MAX is not None:
    lo = f"{MJD_MIN:.4f}" if MJD_MIN is not None else "-∞"
    hi = f"{MJD_MAX:.4f}" if MJD_MAX is not None else "+∞"
    print(f"MJD filter   : [{lo},  {hi}]")
else:
    print("MJD filter   : none")

## 2. Utility functions

In [ ]:
def flux_to_mag_AB(flux_nJy):
    """Convert flux in nJy to AB magnitude. Returns NaN for non-positive flux."""
    try:
        f = float(flux_nJy)
    except (TypeError, ValueError):
        return np.nan
    return -2.5 * np.log10(f / AB_FLUX_ZERO) if (np.isfinite(f) and f > 0) else np.nan


def flux_to_luptitude(flux_nJy, b_soft=None):
    """Convert flux (nJy) to asinh magnitude (Luptitude, Lupton 1999).
    If b_soft is None it defaults to median(|flux|)/3.
    """
    f = np.asarray(flux_nJy, dtype=float)
    if b_soft is None or not np.isfinite(b_soft) or b_soft <= 0:
        b_soft = max(float(np.nanmedian(np.abs(f))) / 3.0, 1.0)
    log10_e_inv = 1.085736
    lup = -2.5 * log10_e_inv * (np.arcsinh(f / (2.0 * b_soft)) + np.log(b_soft)) + (
        31.4 + 2.5 * np.log10(b_soft)
    )
    return lup, b_soft


def zscale(arr, lo=1.0, hi=99.0):
    """Return (vmin, vmax) from percentile-based z-scale stretch."""
    finite = arr[np.isfinite(arr)]
    if len(finite) == 0:
        return 0.0, 1.0
    return float(np.percentile(finite, lo)), float(np.percentile(finite, hi))


def symvlim(arr, percentile=99.5):
    """Return symmetric vmax for a diverging (difference) image display."""
    finite = arr[np.isfinite(arr)]
    v = float(np.percentile(np.abs(finite), percentile)) if len(finite) else 1.0
    return max(v, 1e-9)


def load_npy(src_id, band, kind):
    """Load fullcutouts_{obj}/cutouts/{src_id}_{band}_{kind}.npy.
    Returns a float32 array or None if the file is missing.
    """
    fpath = os.path.join(DIR_CUTOUTS, "cutouts", f"{src_id}_{band}_{kind}.npy")
    return np.load(fpath).astype(np.float32) if os.path.exists(fpath) else None


def parse_bool(val):
    """Coerce a dipole flag (bool / int / str / NaN) to Python bool."""
    if isinstance(val, bool):
        return val
    if isinstance(val, (int, float)) and np.isfinite(float(val)):
        return bool(int(val))
    if isinstance(val, str):
        return val.strip().lower() in ("true", "1", "yes")
    return False


def mjd_to_date(mjd):
    """MJD (TAI) → ISO date string YYYY-MM-DD."""
    try:
        return Time(float(mjd), format="mjd", scale="tai").isot[:10]
    except Exception:
        return "?"


def draw_crosshair(ax, arr):
    """Draw a centred crosshair on an image axis."""
    if arr is None:
        return
    ny, nx = arr.shape
    ax.axvline(nx / 2, color="yellow", lw=0.9, ls="--", alpha=0.8)
    ax.axhline(ny / 2, color="yellow", lw=0.9, ls="--", alpha=0.8)


def fmt_flag(val, true_str="✓", false_str="✗"):
    """Format a boolean dipole flag as a unicode symbol."""
    return true_str if parse_bool(val) else false_str


def fmt_float(val, fmt=".3f", na="—"):
    """Format a float; return *na* string for NaN/None."""
    try:
        v = float(val)
        return f"{v:{fmt}}" if np.isfinite(v) else na
    except (TypeError, ValueError):
        return na


def getminmaxvals(vals):
    """ """
    # --- robust y-limits from diaSource flux, excluding outliers ---
    y = vals.dropna()
    if len(y) >= 5:
        q1, q3 = y.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr > 0:
            lo = q1 - 1.5 * iqr
            hi = q3 + 1.5 * iqr
            y_in = y[(y >= lo) & (y <= hi)]
        else:
            y_in = y

        if len(y_in) > 0:
            ymin, ymax = y_in.min(), y_in.max()
            pad = 0.5 * (ymax - ymin) if ymax > ymin else max(1.0, 0.1 * abs(ymax))

        return ymin - pad, ymax + pad


print("Utility functions defined.")

## 3. Load object-level dipole statistics from notebook 03b

In [ ]:
# ── Object-level dipole catalogue (from topranked_objects_dipoles.csv) ────────
# Columns: diaObjectId, field, nDiaSources, n_src, n_dipoles, dipole_fraction,
#          ra, dec, gaia_name, simbad, label, label_clf,
#          n_dip_u, n_dip_g, n_dip_r, n_dip_i, n_dip_z, n_dip_y, rank

obj_field = "?"
obj_ra, obj_dec = np.nan, np.nan
obj_nDiaSources = 0  # total detections reported by the AP pipeline
obj_n_src = 0  # diaSources fetched via /api/v1/sources
obj_n_dipoles = 0  # diaSources with isDipole=True
obj_dipole_frac = np.nan  # = n_dipoles / n_src
obj_rank = "?"
obj_label = "?"
obj_label_clf = "?"
obj_gaia_name = "?"
obj_simbad = "?"
obj_n_dip_per_band = {}  # dict band → n_dipoles

if os.path.exists(FILE_DIPOLE_STATS):
    df_stats = pd.read_csv(FILE_DIPOLE_STATS)
    hit = df_stats[df_stats["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        obj_field = str(r.get("field", "?"))
        obj_ra = float(r.get("ra", np.nan))
        obj_dec = float(r.get("dec", np.nan))
        obj_nDiaSources = int(r.get("nDiaSources", 0))
        obj_n_src = int(r.get("n_src", 0))
        obj_n_dipoles = int(r.get("n_dipoles", 0))
        obj_dipole_frac = float(r.get("dipole_fraction", np.nan))
        obj_rank = str(r.get("rank", "?"))
        obj_label = str(r.get("label", "?"))
        obj_label_clf = str(r.get("label_clf", "?"))
        obj_gaia_name = str(r.get("gaia_name", "?"))
        obj_simbad = str(r.get("simbad", "?"))
        for band in BAND_ORDER:
            col = f"n_dip_{band}"
            obj_n_dip_per_band[band] = int(r.get(col, 0)) if col in r.index else 0
    else:
        print(f"  ⚠ diaObjectId {DIAOBJECT_ID} not found in {FILE_DIPOLE_STATS}")
else:
    print(f"  ⚠ {FILE_DIPOLE_STATS} not found — run notebook 03b first.")

print(f"Object : {DIAOBJECT_ID}")
print(f"  Field           : {obj_field}")
print(f"  RA / Dec        : {obj_ra:.5f} / {obj_dec:.5f}")
print(f"  nDiaSources(AP) : {obj_nDiaSources}")
print(f"  n_src (fetched) : {obj_n_src}")
print(f"  n_dipoles       : {obj_n_dipoles}  ({100 * obj_dipole_frac:.1f}%)")
print(f"  rank            : {obj_rank}")
print(f"  Gaia            : {obj_gaia_name}")
print(f"  Simbad          : {obj_simbad}")
print(f"  label / clf     : {obj_label} / {obj_label_clf}")
print(f"  Dipoles per band: {obj_n_dip_per_band}")

In [ ]:
# ── Dipole angle & length stability (from dipole_angle_stability.csv) ─────────
# Columns: diaObjectId, n_dipoles, field,
#          angle_mean_deg, angle_circ_std_deg, length_median_arcsec,
#          angle_cstd_u, angle_cstd_g, angle_cstd_r, angle_cstd_i, angle_cstd_z, angle_cstd_y

obj_angle_mean_deg = np.nan  # mean dipole position angle (deg)
obj_angle_circ_std_deg = np.nan  # circular std of position angle (deg)
obj_length_median_arcsec = np.nan  # median dipole lobe separation (arcsec)
obj_angle_cstd_per_band = {}  # dict band → circular std (deg)

if os.path.exists(FILE_DIPOLE_ANGLES):
    df_ang = pd.read_csv(FILE_DIPOLE_ANGLES)
    hit = df_ang[df_ang["diaObjectId"].astype(str) == str(DIAOBJECT_ID)]
    if len(hit):
        r = hit.iloc[0]
        obj_angle_mean_deg = float(r.get("angle_mean_deg", np.nan))
        obj_angle_circ_std_deg = float(r.get("angle_circ_std_deg", np.nan))
        obj_length_median_arcsec = float(r.get("length_median_arcsec", np.nan))
        for band in BAND_ORDER:
            col = f"angle_cstd_{band}"
            try:
                obj_angle_cstd_per_band[band] = float(r.get(col, np.nan))
            except (TypeError, ValueError):
                obj_angle_cstd_per_band[band] = np.nan
    else:
        print(f"  ⚠ diaObjectId {DIAOBJECT_ID} not found in {FILE_DIPOLE_ANGLES}")
else:
    print(f"  ⚠ {FILE_DIPOLE_ANGLES} not found — run notebook 03b first.")

print(f"Dipole geometry for {DIAOBJECT_ID}:")
print(f"  angle mean      : {fmt_float(obj_angle_mean_deg)} deg")
print(f"  angle circ std  : {fmt_float(obj_angle_circ_std_deg)} deg")
print(f"  length median   : {fmt_float(obj_length_median_arcsec, '.4f')} arcsec")
print(f"  angle cstd/band : {obj_angle_cstd_per_band}")

## 4. Load manifest (diaSource flux + dipole flags)

In [ ]:
if not os.path.exists(FILE_MANIFEST):
    raise FileNotFoundError(
        f"{FILE_MANIFEST} not found.\n"
        f"Run: python fink_download_full_cutouts.py --obj_id {DIAOBJECT_ID}\n"
        "or run notebook 12a_downloadSelectedCutouts.ipynb first."
    )

df = pd.read_csv(FILE_MANIFEST)

# ── Parse boolean dipole flags ────────────────────────────────────────────────
for bool_col in ["r:isDipole", "r:isNegative", "r:dipoleFitAttempted"]:
    if bool_col in df.columns:
        df[bool_col] = df[bool_col].fillna(False).apply(parse_bool)

# Convenience alias (no prefix) used in plotting
df["isDipole"] = df["r:isDipole"] if "r:isDipole" in df.columns else False
df["isNegative"] = df["r:isNegative"] if "r:isNegative" in df.columns else False
df["dipoleFitAttempted"] = df["r:dipoleFitAttempted"] if "r:dipoleFitAttempted" in df.columns else False

# ── Sort by time ──────────────────────────────────────────────────────────────
df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)

# ── Optional MJD window filter ────────────────────────────────────────────────
mask = pd.Series([True] * len(df), index=df.index)
if MJD_MIN is not None:
    mask &= df["r:midpointMjdTai"] >= MJD_MIN
if MJD_MAX is not None:
    mask &= df["r:midpointMjdTai"] <= MJD_MAX
n_before = len(df)
df = df[mask].reset_index(drop=True)
if n_before != len(df):
    print(f"MJD filter: {n_before} → {len(df)} diaSources")

# ── Band filter ───────────────────────────────────────────────────────────────
if BANDS_FILTER is not None:
    df = df[df["r:band"].isin(BANDS_FILTER)].reset_index(drop=True)

n_dip = df["isDipole"].sum()
print(f"Manifest : {len(df)} diaSources  |  {n_dip} isDipole ({100 * n_dip / max(len(df), 1):.1f}%)")
print(f"Bands    : {sorted(df['r:band'].unique())}")
df[
    [
        "r:diaSourceId",
        "r:band",
        "r:visit",
        "r:detector",
        "r:psfFlux",
        "r:scienceFlux",
        "r:templateFlux",
        "isDipole",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleChi2",
    ]
].head()

## 5. Load forced photometry

In [ ]:
df_fp = pd.DataFrame()
if os.path.exists(FILE_MANIFEST_FP):
    df_fp = pd.read_csv(FILE_MANIFEST_FP).sort_values("r:midpointMjdTai").reset_index(drop=True)
    print(f"Forced photometry : {len(df_fp)} points  bands: {sorted(df_fp['r:band'].unique())}")
else:
    print(f"  ⚠ {FILE_MANIFEST_FP} not found — light curve panels will show diaSources only.")

# Reference epoch for relative-time axis
all_mjd = df["r:midpointMjdTai"].values
t0 = float(np.nanmin(all_mjd)) if len(all_mjd) else 0.0
print(f"Time origin t0 = MJD {t0:.4f}  ({mjd_to_date(t0)})")

## 6. Dipole geometry overview (per-object summary)

In [ ]:
# Summary statistics on per-source dipole geometry columns
geo_cols = [
    "r:dipoleFluxDiff",
    "r:dipoleFluxDiffErr",
    "r:dipoleMeanFlux",
    "r:dipoleMeanFluxErr",
    "r:dipoleLength",
    "r:dipoleAngle",
    "r:dipoleNdata",
    "r:dipoleChi2",
]
available_geo = [c for c in geo_cols if c in df.columns]

df_dip = df[df["isDipole"]]
print(f"Dipole-flagged diaSources in manifest: {len(df_dip)}")
if len(df_dip) and available_geo:
    print()
    print(df_dip[available_geo].describe().T.to_string())

In [ ]:
# Dipole length and angle distributions per band
if "r:dipoleLength" in df.columns and "r:dipoleAngle" in df.columns:
    bands_present = [b for b in BAND_ORDER if b in df["r:band"].unique()]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    for band in bands_present:
        sub = df_dip[df_dip["r:band"] == band]
        col = BAND_COLORS.get(band, "gray")
        lengths = sub["r:dipoleLength"].dropna()
        angles = sub["r:dipoleAngle"].dropna()
        if len(lengths):
            axes[0].hist(lengths, bins=30, alpha=0.5, color=col, label=band, density=True)
        if len(angles):
            axes[1].hist(angles, bins=36, alpha=0.5, color=col, label=band, density=True)

    axes[0].set_xlabel("dipoleLength (arcsec)")
    axes[0].set_ylabel("density")
    axes[0].set_title("Dipole lobe separation")
    axes[0].axvline(
        obj_length_median_arcsec,
        color="k",
        ls="--",
        lw=1.2,
        label=f"median={fmt_float(obj_length_median_arcsec, '.3f')}",
    )
    axes[0].legend(fontsize=7)

    axes[1].set_xlabel("dipoleAngle (deg)")
    axes[1].set_ylabel("density")
    axes[1].set_title("Dipole position angle")
    axes[1].axvline(
        obj_angle_mean_deg, color="k", ls="--", lw=1.2, label=f"mean={fmt_float(obj_angle_mean_deg)}°"
    )
    axes[1].legend(fontsize=7)

    fig.suptitle(
        f"diaObjectId {DIAOBJECT_ID}  |  field={obj_field}  "
        f"dipole_frac={fmt_float(obj_dipole_frac, '.3f')}  "
        f"circ_std={fmt_float(obj_angle_circ_std_deg)}°",
        fontsize=12,
    )
    fig.tight_layout()
    if SAVE_FIGS:
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(DIR_FIGS, f"dipole_geometry_{DIAOBJECT_ID}.{ext}"), bbox_inches="tight")
    plt.show()

## 7. Cutout grid — 2×3 layout per diaSource

Each figure shows one diaSource:
- **(0,0)** Info panel: astrometry, fluxes, dipole geometry (length, angle, chi²), flags
- **(0,1)** Science stamp
- **(0,2)** Template stamp (same colour scale as Science)
- **(1,0)** Light curve in the current band (diaSources + fp, visit highlighted)
- **(1,1)** DIA Difference stamp (downloaded)
- **(1,2)** Sci − Template (computed inline)

In [ ]:
def build_info_text(row):
    """Assemble the multi-line info string for the top-left panel."""
    src_id = int(row["r:diaSourceId"])
    band = row["r:band"]
    mjd = row["r:midpointMjdTai"]
    visit = row.get("r:visit", "?")
    det = row.get("r:detector", "?")

    psf_flux = fmt_float(row.get("r:psfFlux"), ".1f")
    sci_flux = fmt_float(row.get("r:scienceFlux"), ".1f")
    tpl_flux = fmt_float(row.get("r:templateFlux"), ".1f")
    snr = fmt_float(row.get("r:snr"), ".1f")
    reliability = fmt_float(row.get("r:reliability"), ".3f")

    # Dipole flags
    is_dip = fmt_flag(row.get("r:isDipole", False))
    is_neg = fmt_flag(row.get("r:isNegative", False))
    fit_att = fmt_flag(row.get("r:dipoleFitAttempted", False))

    # Dipole geometry
    dip_fd = fmt_float(row.get("r:dipoleFluxDiff"), ".1f")
    dip_mf = fmt_float(row.get("r:dipoleMeanFlux"), ".1f")
    dip_len = fmt_float(row.get("r:dipoleLength"), ".3f")
    dip_ang = fmt_float(row.get("r:dipoleAngle"), ".1f")
    dip_chi2 = fmt_float(row.get("r:dipoleChi2"), ".2f")
    dip_n = int(row.get("r:dipoleNdata", 0)) if pd.notna(row.get("r:dipoleNdata")) else 0

    mag_psf = flux_to_mag_AB(row.get("r:psfFlux"))
    mag_str = f"{mag_psf:.3f}" if np.isfinite(mag_psf) else "—"

    lines = [
        f"diaSourceId: {src_id}",
        f"diaObjectId: {DIAOBJECT_ID}",
        f"band={band}  visit={visit}  det={det}",
        f"MJD={mjd:.4f}  ({mjd_to_date(mjd)})",
        f"RA={fmt_float(row.get('r:ra'), '.5f')}  Dec={fmt_float(row.get('r:dec'), '.5f')}",
        "",
        f"psfFlux   = {psf_flux} nJy  (mag={mag_str})",
        f"sciFlux   = {sci_flux} nJy",
        f"tplFlux   = {tpl_flux} nJy",
        f"SNR       = {snr}  |  reliability = {reliability}",
        "",
        f"isDipole={is_dip}  isNeg={is_neg}  fitAtt={fit_att}",
        f"dipoleFluxDiff  = {dip_fd} nJy",
        f"dipoleMeanFlux  = {dip_mf} nJy",
        f"dipoleLength    = {dip_len} arcsec",
        f"dipoleAngle     = {dip_ang} deg",
        f"dipoleChi2      = {dip_chi2}  (Ndata={dip_n})",
        "",
        f"field={obj_field}  rank={obj_rank}",
        f"dipole_frac={fmt_float(obj_dipole_frac, '.3f')}",
        f"Gaia: {obj_gaia_name}",
        f"Simbad: {obj_simbad}",
    ]
    return "\n".join(lines)


print("Info text builder defined.")

In [ ]:
def plot_lightcurve(ax, band, current_mjd):
    """Plot psfFlux light curve (diaSources + fp) for one band,
    highlighting the current visit with a vertical line.
    """
    # --- diaSources (detections) ---
    sub = df[df["r:band"] == band].sort_values("r:midpointMjdTai")
    col = BAND_COLORS.get(band, "gray")

    ax.errorbar(
        sub["r:midpointMjdTai"] - t0,
        sub["r:psfFlux"],
        yerr=sub.get("r:psfFluxErr", None),
        fmt="o",
        color=col,
        ms=3,
        lw=0.8,
        alpha=0.7,
        label="diaSource",
    )

    ymin, ymax = getminmaxvals(sub["r:psfFlux"])

    # --- forced photometry ---
    if not df_fp.empty:
        fp_band = df_fp[df_fp["r:band"] == band].sort_values("r:midpointMjdTai")
        ax.errorbar(
            fp_band["r:midpointMjdTai"] - t0,
            fp_band["r:psfFlux"],
            yerr=fp_band.get("r:psfFluxErr", None),
            fmt="^",
            color=col,
            ms=4,
            lw=0.6,
            alpha=0.35,
            label="fp",
        )

    # --- current visit marker ---
    ax.axvline(current_mjd - t0, color="k", lw=1.0, ls="--", alpha=0.8)

    ax.set_xlabel(f"MJD − {t0:.0f}  [days]", fontsize=10)
    ax.set_ylabel("psfFlux [nJy]", fontsize=10)
    ax.set_title(f"band={band}", fontsize=10)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8, loc="upper left")
    ax.set_ylim(ymin, ymax)


print("Light-curve helper defined.")

In [ ]:
# définir l’ordre physique des bandes LSST
band_order = ["u", "g", "r", "i", "z", "y"]

# convertir en type catégoriel ordonné
df["r:band"] = pd.Categorical(df["r:band"], categories=BAND_ORDER, ordered=True)

# tri
df_sorted = df.sort_values(by=["r:band", "r:visit"])

In [ ]:
df = df_sorted

In [ ]:
combined

In [ ]:
# ── Apply MAX_ROWS_PER_BAND cap ───────────────────────────────────────────────
if MAX_ROWS_PER_BAND is not None:
    df_plot = (
        df.groupby("r:band", group_keys=False)
        .apply(lambda g: g.head(MAX_ROWS_PER_BAND))
        .reset_index(drop=True)
    )
else:
    df_plot = df

print(f"Plotting {len(df_plot)} diaSources ...")

for idx, row in df_plot.iterrows():
    src_id = int(row["r:diaSourceId"])
    band = row["r:band"]
    mjd = row["r:midpointMjdTai"]
    is_dip = parse_bool(row.get("r:isDipole", False))

    # Load stamps
    arr_sci = load_npy(src_id, band, "Science")
    arr_tpl = load_npy(src_id, band, "Template")
    arr_diff = load_npy(src_id, band, "Difference")

    if arr_sci is None and arr_tpl is None and arr_diff is None:
        print(f"  [{idx + 1:3d}] {src_id} {band} — no .npy files found, skipping.")
        continue

    # Compute Sci − Tpl inline
    if arr_sci is not None and arr_tpl is not None:
        arr_sci_minus_tpl = arr_sci - arr_tpl
    else:
        arr_sci_minus_tpl = None

    # ── Layout ─────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(13, 6))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.25)
    ax_info = fig.add_subplot(gs[0, 0])
    ax_sci = fig.add_subplot(gs[0, 1])
    ax_tpl = fig.add_subplot(gs[0, 2])
    ax_lc = fig.add_subplot(gs[1, 0])
    ax_diff = fig.add_subplot(gs[1, 1])
    ax_smtpl = fig.add_subplot(gs[1, 2])

    # ── (0,0) Info panel ───────────────────────────────────────────────────
    ax_info.axis("off")
    dipole_color = "#c0392b" if is_dip else "#2c3e50"
    ax_info.text(
        0.04,
        0.97,
        build_info_text(row),
        transform=ax_info.transAxes,
        va="top",
        ha="left",
        fontsize=7,
        fontfamily="monospace",
        color=dipole_color,
    )

    # ── (0,1) Science ──────────────────────────────────────────────────────
    if arr_sci is not None:
        combined = (
            np.concatenate([arr_sci.ravel(), arr_tpl.ravel()]) if arr_tpl is not None else arr_sci.ravel()
        )
        vmin_st, vmax_st = zscale(combined)
        norm_st = Normalize(vmin=vmin_st, vmax=vmax_st)
        im_sci = ax_sci.imshow(arr_sci, origin="lower", cmap=CMAP_SCI, norm=norm_st)
        draw_crosshair(ax_sci, arr_sci)
    else:
        ax_sci.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_sci.transAxes)

    ax_sci.set_title(f"Science  {band}  MJD={mjd:.3f}", fontsize=8)
    ax_sci.axis("off")

    # ── (0,2) Template ─────────────────────────────────────────────────────
    if arr_tpl is not None:
        im_tpl = ax_tpl.imshow(arr_tpl, origin="lower", cmap=CMAP_SCI, norm=norm_st)
        draw_crosshair(ax_tpl, arr_tpl)
    else:
        ax_tpl.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_tpl.transAxes)

    fig.colorbar(im_sci, ax=[ax_sci, ax_tpl], fraction=0.046, pad=0.04)

    ax_tpl.set_title("Template", fontsize=8)
    ax_tpl.axis("off")

    # ── (1,0) Light curve ──────────────────────────────────────────────────
    plot_lightcurve(ax_lc, band, mjd)

    # ── (1,1) DIA Difference ───────────────────────────────────────────────

    combined = (
        np.concatenate([arr_diff.ravel(), arr_sci_minus_tpl.ravel()])
        if arr_sci_minus_tpl is not None
        else arr_diff.ravel()
    )
    vmax_diff = symvlim(combined)
    norm_diff = Normalize(vmin=-vmax_diff, vmax=vmax_diff)

    if arr_diff is not None:
        # vmax_diff = symvlim(arr_diff)
        # ax_diff.imshow(arr_diff, origin="lower", cmap=CMAP_DIFF, vmin=-vmax_diff, vmax=vmax_diff)

        im_diff = ax_diff.imshow(arr_diff, origin="lower", cmap=CMAP_DIFF, norm=norm_diff)

        draw_crosshair(ax_diff, arr_diff)
        dip_lbl = "  [DIPOLE]" if is_dip else ""
        ax_diff.set_title(f"DIA Difference{dip_lbl}", fontsize=8, color="#c0392b" if is_dip else "black")
    else:
        ax_diff.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_diff.transAxes)
        ax_diff.set_title("DIA Difference", fontsize=8)
    ax_diff.axis("off")

    # ── (1,2) Sci − Template ───────────────────────────────────────────────
    if arr_sci_minus_tpl is not None:
        # vmax_smt = symvlim(arr_sci_minus_tpl)
        # vmax_smt = vmax_diff
        # ax_smtpl.imshow(arr_sci_minus_tpl, origin="lower", cmap=CMAP_DIFF,vmin=-vmax_smt, vmax=vmax_smt)

        im_smtpl = ax_smtpl.imshow(arr_sci_minus_tpl, origin="lower", cmap=CMAP_DIFF, norm=norm_diff)

        draw_crosshair(ax_smtpl, arr_sci_minus_tpl)
    else:
        ax_smtpl.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax_smtpl.transAxes)
    ax_smtpl.set_title("Sci − Template  (computed)", fontsize=8)
    ax_smtpl.axis("off")

    cbar = fig.colorbar(im_diff, ax=[ax_diff, ax_smtpl], fraction=0.046, pad=0.04)
    cbar.set_label("Flux difference")

    # ── Title ──────────────────────────────────────────────────────────────
    dip_str = f"isDipole={is_dip}  L={fmt_float(row.get('r:dipoleLength'), '.3f')}″  θ={fmt_float(row.get('r:dipoleAngle'), '.1f')}°"
    fig.suptitle(
        f"obj={DIAOBJECT_ID}  src={src_id}  [{idx + 1}/{len(df_plot)}]  |  {dip_str}",
        fontsize=14,
        color=BAND_COLORS[band],
    )

    if SAVE_FIGS:
        fname = f"cutout_{DIAOBJECT_ID}_{src_id}_{band}"
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(DIR_FIGS, f"{fname}.{ext}"), bbox_inches="tight", dpi=130)

    plt.show()
    plt.close(fig)

print("Done.")